# 05 — Feature engineering

**Project:** Hotel overbooking decision support  
**Stage:** Turn the selected source columns into a model-ready feature table  
**Purpose:** Create a clear and repeatable table of inputs (`X`) and the outcome (`y`).

Feature engineering means preparing the information that a model will use. It does not mean choosing columns because they happen to improve a score. The columns must first pass the availability and leakage review in Notebook 04.

## What this notebook will do

1. Load the original booking data.
2. Define the candidate columns approved by the current column review.
3. Separate categorical and numeric columns.
4. Handle missing categories without inventing numeric values.
5. keep the target separate from the inputs.
6. Check the resulting feature table.
7. Save a small feature dictionary and evidence files.

**Important:** The supplied CSV is a near-arrival snapshot, not a proven snapshot from the moment each booking was created. The resulting table is therefore an experimental candidate, not yet a production-certified booking-time feature set.

In [1]:
# Run from the repository root, so `from src import ...` and the relative evidence/ paths work
# whether this notebook is opened from the repo folder or from notebooks/pipeline/.
import os
import sys
from pathlib import Path

REPO_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src" / "contract.py").exists())
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
print("Working directory:", REPO_ROOT)

Working directory: C:\Users\Lenovo\Desktop\Build_2_Repo\IIT_GN_W06


## 1. Load the original data

We use the shared project loader. It checks the source dimensions and does not modify the CSV.

In [2]:
import pandas as pd
from pathlib import Path
from IPython.display import display

from src import data

bookings = data.read_raw()

print("Bookings:", f"{len(bookings):,}")
print("Columns:", len(bookings.columns))

Bookings: 119,390
Columns: 32


## 2. Choose the candidate columns

The column review in Notebook 04 identified the following provisional candidates:
- `hotel`
- `market_segment`
- `distribution_channel`
- `is_repeated_guest`
- `previous_cancellations`
- `previous_bookings_not_canceled`
- `agent`
- `company`

We leave `country` out of this working feature table because the source paper warns that nationality may only be known at check-in. The other candidate fields still need timestamp validation against the hotel's booking system before production use.

In [3]:
categorical_columns = [
    "hotel",
    "market_segment",
    "distribution_channel",
    "agent",
    "company"
]

numeric_columns = [
    "is_repeated_guest",
    "previous_cancellations",
    "previous_bookings_not_canceled"
]

feature_columns = categorical_columns + numeric_columns

print("Categorical columns:", categorical_columns)
print("Numeric columns:", numeric_columns)
print("Total candidate features:", len(feature_columns))

Categorical columns: ['hotel', 'market_segment', 'distribution_channel', 'agent', 'company']
Numeric columns: ['is_repeated_guest', 'previous_cancellations', 'previous_bookings_not_canceled']
Total candidate features: 8


## 3. Check that the selected columns exist

This check prevents a spelling mistake or a changed source file from silently creating an incomplete feature table.

In [4]:
missing_columns = [
    column for column in feature_columns
    if column not in bookings.columns
]

print("Missing selected columns:", missing_columns)

assert missing_columns == [], "One or more selected columns are not in the source data."

print("Check passed: all selected feature columns exist.")

Missing selected columns: []
Check passed: all selected feature columns exist.


## 4. Create the feature table

`X` contains only the selected input columns. We make a copy so that any preparation does not change the original booking table.

The target is deliberately not included in `X`.

In [5]:
X = bookings[feature_columns].copy()

print("Feature table shape:", X.shape)
display(X.head())

Feature table shape: (119390, 8)


,hotel,market_segment,distribution_channel,agent,company,is_repeated_guest,previous_cancellations,previous_bookings_not_canceled
0,Resort Hotel,Direct,Direct,NaN,NaN,0,0,0
1,Resort Hotel,Direct,Direct,NaN,NaN,0,0,0
2,Resort Hotel,Direct,Direct,NaN,NaN,0,0,0
3,Resort Hotel,Corporate,Corporate,304.0,NaN,0,0,0
4,Resort Hotel,Online TA,TA/TO,240.0,NaN,0,0,0


## 5. Treat categorical columns as categories

A value such as agent ID agent 9 is not necessarily greater than agent ID 3 in any meaningful business sense. These identifiers are labels, not quantities.

We convert these fields to text categories. We also make missing values explicit:
- missing agent becomes `NO_AGENT`;
- missing company becomes `NO_COMPANY`;
- missing hotel, segment or channel gets an `UNKNOWN_...` label.

This preserves the difference between “no recorded agent” and a numeric value of zero.

In [6]:
for column in categorical_columns:
    X[column] = X[column].astype("string")

X["agent"] = X["agent"].fillna("NO_AGENT")
X["company"] = X["company"].fillna("NO_COMPANY")

for column in ["hotel", "market_segment", "distribution_channel"]:
    X[column] = X[column].fillna("UNKNOWN_" + column.upper())

display(X[categorical_columns].head())

,hotel,market_segment,distribution_channel,agent,company
0,Resort Hotel,Direct,Direct,NO_AGENT,NO_COMPANY
1,Resort Hotel,Direct,Direct,NO_AGENT,NO_COMPANY
2,Resort Hotel,Direct,Direct,NO_AGENT,NO_COMPANY
3,Resort Hotel,Corporate,Corporate,304.0,NO_COMPANY
4,Resort Hotel,Online TA,TA/TO,240.0,NO_COMPANY


## 6. Check the numeric columns

These three columns are numeric counts or flags. We do not fill missing values with zero because zero has a real meaning.

If any of these fields is missing, stop and investigate rather than silently changing the record.

In [7]:
for column in numeric_columns:
    print(column, "missing values:", X[column].isna().sum())

assert X[numeric_columns].notna().all().all(), (
    "A selected numeric feature contains missing values. Investigate before modelling."
)

display(X[numeric_columns].describe().T)

is_repeated_guest missing values: 0
previous_cancellations missing values: 0
previous_bookings_not_canceled missing values: 0


,count,mean,std,min,25%,50%,75%,max
is_repeated_guest,119390.0,0.031912,0.175767,0.0,0.0,0.0,0.0,1.0
previous_cancellations,119390.0,0.087118,0.844336,0.0,0.0,0.0,0.0,26.0
previous_bookings_not_canceled,119390.0,0.137097,1.497437,0.0,0.0,0.0,0.0,72.0


## 7. Create the target separately

The target is `is_canceled`:
- 1 means cancelled or no-show;
- 0 means the guest arrived.

We create `y` separately. This makes it harder to accidentally include the answer inside the model inputs.

In [8]:
y = bookings[data.TARGET].copy()

print("Target name:", data.TARGET)
print("Target values:")
display(y.value_counts().sort_index().to_frame("bookings"))

assert set(y.unique()).issubset({0, 1})
assert len(X) == len(y)

print("Check passed: every feature row has one target value.")

Target name: is_canceled
Target values:


,bookings
is_canceled,
0,75166
1,44224


Check passed: every feature row has one target value.

## 8. Check the final feature table

Before using the table in a model, check:
- the row count has not changed;
- the feature names are exactly the selected names;
- there are no target or final-outcome columns in the feature table;
- categorical columns contain text labels;
- numeric columns remain numeric.

In [9]:
forbidden_columns = [
    "is_canceled",
    "reservation_status",
    "reservation_status_date",
    "booking_changes",
    "assigned_room_type"
]

print("Feature rows:", len(X))
print("Target rows:", len(y))
print("Feature columns:", list(X.columns))
print("Forbidden columns present:", [c for c in forbidden_columns if c in X.columns])

assert len(X) == len(bookings)
assert len(y) == len(bookings)
assert list(X.columns) == feature_columns
assert not any(column in X.columns for column in forbidden_columns)
assert all(X[column].dtype.name in ["string", "object"] for column in categorical_columns)
assert all(pd.api.types.is_numeric_dtype(X[column]) for column in numeric_columns)

print("All feature-table checks passed.")

Feature rows: 119390
Target rows: 119390
Feature columns: ['hotel', 'market_segment', 'distribution_channel', 'agent', 'company', 'is_repeated_guest', 'previous_cancellations', 'previous_bookings_not_canceled']
Forbidden columns present: []
All feature-table checks passed.


## 9. Understand what encoding means

Most machine-learning models cannot directly work with text labels. A common approach is one-hot encoding: create a separate 0/1 column for each category.

We do not manually create those columns here. We keep the feature table readable and let the model pipeline perform encoding inside each training fold.

This is important because the encoder must learn categories from training data only. If we encode the full dataset before splitting, information about categories in later periods can leak into the training process.

## 10. Create a feature dictionary

A feature dictionary explains each input to a reviewer. It records the business meaning and the reason the feature is included as a candidate.

In [10]:
feature_dictionary = pd.DataFrame([
    {
        "feature": "hotel",
        "type": "categorical",
        "meaning": "Which hotel the booking belongs to",
        "reason": "The two properties may have different cancellation patterns",
        "timing_status": "Candidate; confirm at booking timestamp"
    },
    {
        "feature": "market_segment",
        "type": "categorical",
        "meaning": "Commercial market segment associated with the booking",
        "reason": "Different booking segments may behave differently",
        "timing_status": "Provisional; confirm it is not reclassified later"
    },
    {
        "feature": "distribution_channel",
        "type": "categorical",
        "meaning": "Channel through which the booking was made",
        "reason": "Different booking channels may have different cancellation behaviour",
        "timing_status": "Provisional; confirm it is not reclassified later"
    },
    {
        "feature": "agent",
        "type": "categorical",
        "meaning": "Travel agent identifier, when present",
        "reason": "Agent identifiers are labels, not ordered numbers",
        "timing_status": "Provisional; missing means no recorded agent"
    },
    {
        "feature": "company",
        "type": "categorical",
        "meaning": "Company identifier, when present",
        "reason": "Company identifiers are labels, not ordered numbers",
        "timing_status": "Provisional; missing means no recorded company"
    },
    {
        "feature": "is_repeated_guest",
        "type": "numeric flag",
        "meaning": "Whether the guest is recorded as a repeat guest",
        "reason": "Prior guest history may relate to cancellation behaviour",
        "timing_status": "Provisional; verify history was known before this booking"
    },
    {
        "feature": "previous_cancellations",
        "type": "numeric count",
        "meaning": "Number of previous cancellations",
        "reason": "Past cancellation behaviour may be informative",
        "timing_status": "Provisional; verify count uses only earlier events"
    },
    {
        "feature": "previous_bookings_not_canceled",
        "type": "numeric count",
        "meaning": "Number of previous bookings that were not cancelled",
        "reason": "Past successful stays may be informative",
        "timing_status": "Provisional; verify count uses only earlier events"
    }
])

display(feature_dictionary)

,feature,type,meaning,reason,timing_status
0,hotel,categorical,Which hotel the booking belongs to,The two properties may have different cancella...,Candidate; confirm at booking timestamp
1,market_segment,categorical,Commercial market segment associated with the ...,Different booking segments may behave differently,Provisional; confirm it is not reclassified later
2,distribution_channel,categorical,Channel through which the booking was made,Different booking channels may have different ...,Provisional; confirm it is not reclassified later
3,agent,categorical,"Travel agent identifier, when present","Agent identifiers are labels, not ordered numbers",Provisional; missing means no recorded agent
4,company,categorical,"Company identifier, when present","Company identifiers are labels, not ordered nu...",Provisional; missing means no recorded company
5,is_repeated_guest,numeric flag,Whether the guest is recorded as a repeat guest,Prior guest history may relate to cancellation...,Provisional; verify history was known before t...
6,previous_cancellations,numeric count,Number of previous cancellations,Past cancellation behaviour may be informative,Provisional; verify count uses only earlier ev...
7,previous_bookings_not_canceled,numeric count,Number of previous bookings that were not canc...,Past successful stays may be informative,Provisional; verify count uses only earlier ev...


## 11. Save the feature evidence

We save:
- the feature dictionary;
- the list of feature names;
- a small sample of the feature table with its target for inspection.

The full feature table is not saved as a new source dataset. It can be recreated from the original CSV by running this notebook.

In [11]:
evidence_folder = Path("evidence/feature_engineering")
evidence_folder.mkdir(parents=True, exist_ok=True)

feature_dictionary.to_csv(
    evidence_folder / "feature_dictionary.csv",
    index=False
)

pd.DataFrame({"feature": X.columns}).to_csv(
    evidence_folder / "selected_feature_columns.csv",
    index=False
)

sample = X.head(20).copy()
sample["is_canceled"] = y.head(20).values

sample.to_csv(
    evidence_folder / "feature_table_sample.csv",
    index=False
)

print("Saved feature-engineering evidence to:", evidence_folder)

Saved feature-engineering evidence to: evidence\feature_engineering


## 12. Important discrepancy to resolve before model training

The existing `src/features.py` in the starter project currently includes `country` in its feature list. This notebook deliberately holds `country` back because Notebook 04 identified a timing concern from the source paper.

Do not silently ignore this difference. Before training:
1. Review the actual `src/features.py`.
2. Decide with the Data Lead whether `country` is excluded until booking-time availability is proven.
3. Update the reusable feature code to match the approved feature list.
4. Add a test that fails if a held-back field is included.

The notebook is the analytical record; `src/features.py` must eventually implement the decision made here.

## 13. Questions you should be able to answer

1. Why do we keep agent and company identifiers as categories rather than numeric values?
2. Why is `NO_AGENT` different from agent ID zero?
3. Why do we keep the target `y` separate from the feature table `X`?
4. Why should one-hot encoding be fitted only on training data?
5. Why is the feature table still only a candidate for production use?

**Next notebook:** `06_baselines_and_pooling_lab.ipynb`, where we establish simple baselines before trying more complex machine-learning models.